In [1]:
import chess, chess.engine, os, stat
from policy import *
import random
from discrim import *

2026-04-14 22:57:53.387865: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-14 22:57:53.389463: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-14 22:57:53.426404: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-14 22:57:53.427044: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-14 22:57:54.882207: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Co

POLICY V7


In [2]:
from stockfish import Stockfish
engine_path = r"./stockfish/src/stockfish"
sf = Stockfish(engine_path, parameters={"Threads": 1, "Hash": 256})
sf.set_depth(2)
sf.set_skill_level(2)
sf.get_engine_parameters()

{'Debug Log File': '',
 'Contempt': 0,
 'Min Split Depth': 0,
 'Ponder': False,
 'MultiPV': 1,
 'Skill Level': 2,
 'Move Overhead': 10,
 'Minimum Thinking Time': 20,
 'Slow Mover': 100,
 'UCI_Chess960': False,
 'UCI_LimitStrength': False,
 'UCI_Elo': 1350,
 'Threads': 1,
 'Hash': 256}

In [3]:
games= load_json("./data/Bijay_1549_games.json")
print(len(games))

Loading games: 100%|██████████| 392/392 [00:00<00:00, 507.74it/s]

392


In [4]:
agent = Agent("Bijay_1549",stockfish_path=r"./stockfish/src/stockfish")
agent.train(games)

Agent V2
[]
Epoch 1/10
396/396 [==============================] - 134s 331ms/step - loss: 0.0026 - accuracy: 0.0872
Epoch 2/10
396/396 [==============================] - 131s 331ms/step - loss: 0.0019 - accuracy: 0.0933
Epoch 3/10
396/396 [==============================] - 131s 331ms/step - loss: 0.0016 - accuracy: 0.0955
Epoch 4/10
396/396 [==============================] - 131s 331ms/step - loss: 0.0014 - accuracy: 0.0953
Epoch 5/10
396/396 [==============================] - 131s 331ms/step - loss: 0.0011 - accuracy: 0.0948
Epoch 6/10
396/396 [==============================] - 133s 335ms/step - loss: 8.1486e-04 - accuracy: 0.0940
Epoch 7/10
396/396 [==============================] - 134s 339ms/step - loss: 5.8577e-04 - accuracy: 0.0946
Epoch 8/10
396/396 [==============================] - 134s 338ms/step - loss: 4.1651e-04 - accuracy: 0.0951
Epoch 9/10
396/396 [==============================] - 132s 333ms/step - loss: 3.0861e-04 - accuracy: 0.0950
Epoch 10/10
396/396 [===============

In [5]:
def simulate_games(agent, sf, num_games=500, file_dir="./data", file_postfix="0"):

    games_data = []

    for i in range(num_games):
        board = chess.Board()
        moves = []

        while not board.is_game_over():
            if board.turn == chess.WHITE:
                move = agent.act(board)
            else:
                sf.set_fen_position(board.fen())
                best = sf.get_best_move()

                if best is None:
                    move = chess.Move.from_uci("0000")
                    break

                move = chess.Move.from_uci(best)

            board.push(move)
            moves.append(move.uci())

        game_data = {
            "event": "Agent vs Stockfish",
            "round": i + 1,
            "white": f"Mimic Agent of {agent.id}",
            "black": "Stockfish",
            "result": board.result(),
            "moves": moves
        }

        games_data.append(game_data)

    # 🔥 normalize postfix here (IMPORTANT)
    file_postfix = str(file_postfix).replace(".", "_")

    file_path = f"{file_dir}/{agent.id}_agent_vs_stockfish_{file_postfix}.json"

    with open(file_path, "w") as f:
        json.dump(games_data, f, indent=4)

    return file_path 

In [6]:
def overall_similarity_pipeline(json_A, json_B, player_A, player_B):

    print(f"\n--- FULL PIPELINE: {player_A} vs {player_B} ---\n")

    # ============================================================
    # 🔧 FIX: normalize JSON INSIDE PIPELINE (list → string)
    # ============================================================
    def normalize_json(path):
        with open(path, "r") as f:
            games = json.load(f)

        for g in games:
            if isinstance(g.get("moves"), list):
                g["moves"] = " ".join(g["moves"])

        return games

    # Write temporary cleaned versions (no external preprocessing step)
    import tempfile

    def write_temp(games):
        tmp = tempfile.NamedTemporaryFile(delete=False, mode="w", suffix=".json")
        json.dump(games, tmp)
        tmp.close()
        return tmp.name

    clean_A = write_temp(normalize_json(json_A))
    clean_B = write_temp(normalize_json(json_B))

    # ============================================================
    # ORIGINAL PIPELINE (UNCHANGED LOGIC BELOW)
    # ============================================================
    b_A, m_A, l_A = load_json_game_sequences(clean_A, player_A, 1.0)
    b_B, m_B, l_B = load_json_game_sequences(clean_B, player_B, 0.0)

    min_games = min(len(l_A), len(l_B))
    if min_games == 0:
        print("Not enough usable games.")
        return None

    b_A, m_A, l_A = b_A[:min_games], m_A[:min_games], l_A[:min_games]
    b_B, m_B, l_B = b_B[:min_games], m_B[:min_games], l_B[:min_games]

    raw_boards = b_A + b_B
    raw_moves  = m_A + m_B
    raw_labels = l_A + l_B

    combined = list(zip(raw_boards, raw_moves, raw_labels))
    random.shuffle(combined)
    raw_boards, raw_moves, raw_labels = zip(*combined)

    all_boards = np.array(raw_boards)
    all_moves  = np.array(raw_moves)
    all_labels = np.array(raw_labels)

    all_moves_onehot = tf.one_hot(all_moves, NUM_MOVES)

    model = build_style_classifier()
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001, clipnorm=1.0),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    )

    model.fit(
        x={"board_seq": all_boards, "move_seq": all_moves_onehot},
        y=all_labels,
        batch_size=32,
        epochs=20,
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=1
    )

    similarity = compute_overall_similarity(
        clean_A, clean_B, player_A, player_B, model
    )

    print(f"Overall playstyle similarity: {similarity:.2f}%")
    os.remove(clean_A)
    os.remove(clean_B)
    return similarity

In [7]:
def hyper_tuning(agent, sf, file_dir="./data"):
    a_values = np.linspace(0, 1, 11)[::-1]

    best_a = None
    best_score = float("-inf")

    player_file_path = f"{file_dir}/{agent.id}_games.json"

    for a in a_values:
        try:
            agent.a = a

            agent_file_path = simulate_games(
                agent,
                sf,
                100,
                file_dir=file_dir,
                file_postfix=f"_a_{a:.2f}"
            )

            score = overall_similarity_pipeline(
                player_file_path,
                agent_file_path,
                f"{agent.id}",
                f"Mimic Agent of {agent.id}"
            )

            print(f"a={a:.2f}, score={score:.3f}")

            if score > best_score:
                best_score = score
                best_a = a

        finally:
            # 🔥 CRITICAL: prevent Colab crashes
            import gc
            tf.keras.backend.clear_session()
            gc.collect()

    print(f"\nBest a: {best_a:.2f} (score={best_score:.3f})")

In [8]:
hyper_tuning(agent,sf)

/storage/home/jmy5612/Desktop/policy.py:162: UserWarning: Note that even though you've set Stockfish to play on a weaker elo or skill level, get_evaluation will still return full strength Stockfish's evaluation of the position.
  eval_info = self.sf.get_evaluation()



--- FULL PIPELINE: Bijay_1549 vs Mimic Agent of Bijay_1549 ---

Epoch 1/20
5/5 [==============================] - 10s 1s/step - loss: 1.8086 - accuracy: 0.5688 - val_loss: 1.7943 - val_accuracy: 0.4500
Epoch 2/20
5/5 [==============================] - 4s 814ms/step - loss: 1.7740 - accuracy: 0.5750 - val_loss: 1.7653 - val_accuracy: 0.4500
Epoch 3/20
5/5 [==============================] - 4s 833ms/step - loss: 1.7312 - accuracy: 0.5625 - val_loss: 1.7259 - val_accuracy: 0.4500
Epoch 4/20
5/5 [==============================] - 4s 825ms/step - loss: 1.6732 - accuracy: 0.6562 - val_loss: 1.6581 - val_accuracy: 0.8250
Epoch 5/20
5/5 [==============================] - 4s 826ms/step - loss: 1.5771 - accuracy: 0.8188 - val_loss: 1.5616 - val_accuracy: 0.8750
Epoch 6/20
5/5 [==============================] - 4s 844ms/step - loss: 1.4491 - accuracy: 0.9062 - val_loss: 1.4367 - val_accuracy: 0.9500
Epoch 7/20
5/5 [==============================] - 4s 891ms/step - loss: 1.3220 - accuracy: 0.9937